## 1. Install Dependencies

In [1]:
!pip install -q google-generativeai datasets transformers \ torch tqdm pandas scikit-learn accelerate groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 11.6 MB/s eta 0:00:00


In [2]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT found")

GPU: Tesla T4


## 2. Upload Scripts

In [3]:
import os
for f in ["synthetic_data_gen.py", "reward_model.py"]: print("✓" if os.path.exists(f) else "✗ MISSING:", f)

✓ synthetic_data_gen.py
✓ reward_model.py


## 3. Set API Key

In [4]:
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

## 4. Test Run — 10 Samples

In [5]:
import json
from synthetic_data_gen import PipelineConfig, SyntheticDataPipeline

config = PipelineConfig(
    groq_api_key=os.environ["GROQ_API_KEY"],
    num_samples=10,   # test first
    output_dir="./rlhf_data_test",
)
pipeline = SyntheticDataPipeline(config)
pipeline.run()

with open("./rlhf_data_test/rlhf_dataset.json") as f:
    data = json.load(f)
s = data[0]
print("PROMPT:", s["prompt"])
print("\nCHOSEN:", s["response_chosen"][:100])
print("\nREJECTED:", s["response_rejected"][:100])
print("\nFEEDBACK:", s["rewrite_feedback"])


  Synthetic RLHF Data Generation Pipeline
  Provider : Groq
  Model    : llama-3.3-70b-versatile
  Samples  : 10



Generating: 100%|██████████| 10/10 [00:23<00:00,  2.36s/it]



✓ Generated: 10 | Failed: 0


Saving the dataset (0/1 shards):   0%|          | 0/9 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

   Train: 9 | Val: 1

✓ Final dataset saved:
   JSON → ./rlhf_data_test/rlhf_dataset.json
   CSV  → ./rlhf_data_test/rlhf_dataset.csv
   HF   → ./rlhf_data_test/hf_dataset/
PROMPT: A car travels 60 mph for 2 hours then 40 mph for 3 hours. What is the average speed?

CHOSEN: To find the average speed, we need to calculate the total distance traveled and then divide it by th

REJECTED: The average speed is something like 50 mph because the car travels at different speeds for different

FEEDBACK: The low-quality response lacks specific calculations and explanations. To improve, provide step-by-step calculations for the total distance traveled and the total time taken. Additionally, clearly state the formula used to calculate the average speed.


## 5. Generate Full Dataset

In [6]:
from synthetic_data_gen import PipelineConfig, SyntheticDataPipeline, print_dataset_stats

config = PipelineConfig(
    groq_api_key=os.environ["GROQ_API_KEY"],
    num_samples=100,
    output_dir="./rlhf_data",
    delay_between_calls=1.0,
)
pipeline = SyntheticDataPipeline(config)
pipeline.run()

print_dataset_stats("./rlhf_data")


  Synthetic RLHF Data Generation Pipeline
  Provider : Groq
  Model    : llama-3.3-70b-versatile
  Samples  : 100



Generating:  50%|█████     | 50/100 [02:09<02:23,  2.87s/it]

  Checkpoint saved → ./rlhf_data/checkpoint_50.json


Generating: 100%|██████████| 100/100 [04:46<00:00,  2.87s/it]

  Checkpoint saved → ./rlhf_data/checkpoint_100.json

✓ Generated: 100 | Failed: 0


Saving the dataset (0/1 shards):   0%|          | 0/90 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

   Train: 90 | Val: 10

✓ Final dataset saved:
   JSON → ./rlhf_data/rlhf_dataset.json
   CSV  → ./rlhf_data/rlhf_dataset.csv
   HF   → ./rlhf_data/hf_dataset/

  Dataset Statistics
  Total samples          : 100
  Domain breakdown       :
    instruction_following          43
    open_ended_qa                  28
    reasoning                      17
    creative_writing               12
  Avg quality (chosen)   : 0.904
  Avg quality (rejected) : 0.284
  Quality gap            : 0.620
  Avg prompt length      : 56 chars
  Avg chosen length      : 551 chars
  Avg feedback length    : 327 chars



## 6. Inspect Dataset

In [7]:
import json, pandas as pd
df = pd.read_csv("./rlhf_data/rlhf_dataset.csv")
print(df[["prompt", "rewrite_feedback", "quality_score_chosen",
          "quality_score_rejected"]].head(3).to_string())

                                                                  prompt                                                                                                                                                                                                                                                                                                                                                                                  rewrite_feedback  quality_score_chosen  quality_score_rejected
0  If all A are B, and some B are C, what can we conclude about A and C?                                                           The low-quality response lacks specificity and clarity in explaining the relationship between A and C. It should be expanded to clearly articulate the logical steps and implications of the given statements. Additionally, it needs to accurately convey the nuances of the subset relationship between B and C and how it affects A.                   0.9               

## 7. Train Baseline Reward Model

In [8]:
from reward_model import RewardModelConfig, RewardModelTrainer, RLHFPairDataset
from torch.utils.data import DataLoader
import json

class BaselineRLHFPairDataset(RLHFPairDataset):
    def __init__(self, samples, tokenizer, max_length):

        super().__init__(samples, tokenizer, max_length, augment_with_rewrite=False)

class BaselineRewardModelTrainer(RewardModelTrainer):
    def _load_data(self):
        with open(self.config.data_path) as f:
            samples = json.load(f)
        split = int(0.9 * len(samples))
        train_samples = samples[:split]
        val_samples   = samples[split:]
        print(f"[BASELINE] Train: {len(train_samples)} | Val: {len(val_samples)}")

        train_dataset = BaselineRLHFPairDataset(train_samples, self.tokenizer, self.config.max_length)
        val_dataset   = BaselineRLHFPairDataset(val_samples,   self.tokenizer, self.config.max_length)

        train_loader = DataLoader(train_dataset, batch_size=self.config.batch_size,
                                  shuffle=True,  num_workers=2)
        val_loader   = DataLoader(val_dataset,   batch_size=self.config.batch_size,
                                  shuffle=False, num_workers=2)
        return train_loader, val_loader

baseline_config = RewardModelConfig(
    data_path="./rlhf_data/rlhf_dataset.json",
    output_dir="./reward_model_baseline",
    learning_rate=1e-5,
    epochs=3,
    batch_size=8,
)
baseline_trainer = BaselineRewardModelTrainer(baseline_config)
baseline_trainer.train()


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...


Epoch 1: train_loss=0.6589 | val_loss=0.5546 | val_acc=1.0000
  ✓ New best model saved (val_acc=1.0000)


Epoch 2/3: 100%|██████████| 12/12 [00:11<00:00,  1.09it/s, loss=0.4569]



Epoch 2: train_loss=0.5354 | val_loss=0.3847 | val_acc=1.0000


Epoch 3/3: 100%|██████████| 12/12 [00:10<00:00,  1.09it/s, loss=0.4320]



Epoch 3: train_loss=0.4474 | val_loss=0.3188 | val_acc=1.0000

✓ Training complete. Best val accuracy: 1.0000


## 8. Train Rewrite Reward Model

In [9]:
from reward_model import RewardModelConfig, RewardModelTrainer

rm_config = RewardModelConfig(
    data_path="./rlhf_data/rlhf_dataset.json",
    output_dir="./reward_model",
    base_model="microsoft/deberta-v3-small",
    epochs=3,
    batch_size=8,
    learning_rate=2e-5,
)
trainer = RewardModelTrainer(rm_config)
trainer.train()


Device: cuda


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Train: 90 | Val: 10


Epoch 1/3: 100%|██████████| 23/23 [00:22<00:00,  1.04it/s, loss=0.2412]



Epoch 1: train_loss=0.5156 | val_loss=0.1260 | val_acc=1.0000
  ✓ New best model saved (val_acc=1.0000)


Epoch 2/3: 100%|██████████| 23/23 [00:22<00:00,  1.02it/s, loss=0.0300]



Epoch 2: train_loss=0.0728 | val_loss=0.0069 | val_acc=1.0000


Epoch 3/3: 100%|██████████| 23/23 [00:22<00:00,  1.00it/s, loss=0.0065]



Epoch 3: train_loss=0.0113 | val_loss=0.0031 | val_acc=1.0000

✓ Training complete. Best val accuracy: 1.0000


## 9. In-Distribution Evaluation

In [10]:
import os, torch
from reward_model import RewardModelConfig, ScalarRewardModel
from transformers import AutoTokenizer

def score_response(model_dir, prompt, response, config):
    model_dir = os.path.abspath(model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = ScalarRewardModel(config)
    model.load_state_dict(torch.load(
        os.path.join(model_dir, "model.pt"), map_location=config.device))
    model.float().to(config.device)
    model.eval()
    text = f"[PROMPT] {prompt} [RESPONSE] {response}"
    enc  = tokenizer(text, max_length=config.max_length, truncation=True,
                     padding="max_length", return_tensors="pt")
    with torch.no_grad():
        score = model(enc["input_ids"].to(config.device),
                      enc["attention_mask"].to(config.device))
    return score.item()

rm_config = RewardModelConfig()

test_cases = [
    {
        "prompt": "Explain gradient descent.",
        "good":   "Gradient descent iteratively updates parameters by moving in the direction of steepest loss decrease, scaled by a learning rate.",
        "bad":    "Gradient descent is when the model learns by going downhill somehow."
    },
    {
        "prompt": "What is the difference between precision and recall?",
        "good":   "Precision = TP/(TP+FP): of all predicted positives, how many were correct. Recall = TP/(TP+FN): of all actual positives, how many were caught. High recall matters in medical diagnosis; high precision in spam filtering.",
        "bad":    "Precision is about being correct and recall is about finding everything."
    },
    {
        "prompt": "How does BERT differ from GPT?",
        "good":   "BERT is a bidirectional encoder pretrained with masked language modeling, making it strong for classification and QA. GPT is a unidirectional decoder pretrained with causal language modeling, making it strong for text generation.",
        "bad":    "BERT and GPT are both language models but they work differently."
    },
]

print(f"{'Prompt':<35} {'Model':<12} {'Good':>8} {'Bad':>8} {'Gap':>8}")
print("-" * 75)
for tc in test_cases:
    for label, model_dir in [("Rewrite", "./reward_model/best_model"),
                               ("Baseline", "./reward_model_baseline/best_model")]:
        g = score_response(model_dir, tc["prompt"], tc["good"], rm_config)
        b = score_response(model_dir, tc["prompt"], tc["bad"],  rm_config)
        print(f"{tc['prompt'][:34]:<35} {label:<12} {g:>8.4f} {b:>8.4f} {g-b:>8.4f}")
    print()

Prompt                              Model            Good      Bad      Gap
---------------------------------------------------------------------------


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


How does BERT differ from GPT?      Baseline      -0.1860  -0.3832   0.1971



## 10. Out-of-Distribution Evaluation

In [12]:
ood_test_cases = [
    {
        "prompt": "Write a haiku about machine learning.",
        "good":   "Weights adjust slowly / Loss descends like autumn leaves / Model learns the truth",
        "bad":    "Data flows through nodes / Computers think and learn fast / AI is the future",

    },
    {
        "prompt": "Explain how a refrigerator works.",
        "good":   "A refrigerator uses a compressor to pressurize refrigerant gas, which then expands and absorbs heat from inside the fridge, cooling it down. The heat is expelled outside via condenser coils.",
        "bad":    "A refrigerator keeps things cold using electricity and some chemical stuff inside.",
    },
    {
        "prompt": "What is the capital of Australia?",
        "good":   "The capital of Australia is Sydney ",
        "bad":    "The capital of Australia is Canberra, not Sydney as commonly assumed. It was purpose-built as a compromise between Sydney and Melbourne when Australia federated in 1901.",
    },
]

print(f"{'Prompt':<35} {'Model':<12} {'Good':>8} {'Bad':>8} {'Gap':>8}")
print("-" * 75)
for tc in ood_test_cases:
    for label, model_dir in [("Rewrite",  "./reward_model/best_model"),
                              ("Baseline", "./reward_model_baseline/best_model")]:
        g = score_response(model_dir, tc["prompt"], tc["good"], rm_config)
        b = score_response(model_dir, tc["prompt"], tc["bad"],  rm_config)
        print(f"{tc['prompt'][:34]:<35} {label:<12} {g:>8.4f} {b:>8.4f} {g-b:>8.4f}")
    print()

Prompt                              Model            Good      Bad      Gap
---------------------------------------------------------------------------


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


What is the capital of Australia?   Baseline      -0.3595  -0.4302   0.0707



## 11. Final Comparison Table

In [17]:
results = [
    # (prompt_short,           type,   rewrite_good, rewrite_bad, baseline_good, baseline_bad)
    ("Gradient descent",     "In-dist", -1.2395, -1.6480, -0.0721, -0.2955),
    ("Precision vs Recall",  "In-dist", -0.2325, -1.6309, -0.0106, -0.1492),
    ("BERT vs GPT",          "In-dist", -1.1165, -1.6072, -0.1860, -0.3832),
    ("Haiku (ML)",           "OOD",     -1.6324, -1.5723, -0.1472, -0.1621),
    ("Refrigerator",         "OOD",     -1.2767, -1.5057, -0.1144, -0.1734),
    ("Australia capital",    "OOD",     -1.4999, -1.6471, -0.3595, -0.4302),
]

print(f"{'Prompt':<22} {'Type':<10} {'Rewrite Gap':>12} {'Baseline Gap':>13} {'Improvement':>12}")
print("=" * 72)

rewrite_gaps, baseline_gaps = [], []
for prompt, ptype, rg, rb, bg, bb in results:
    r_gap = rg - rb
    b_gap = bg - bb
    rewrite_gaps.append(r_gap)
    baseline_gaps.append(b_gap)

    if r_gap > 0 and b_gap > 0:
        improvement = f"+{((r_gap - b_gap) / b_gap) * 100:.0f}%"
    elif r_gap <= 0:
        improvement = "failed"
    else:
        improvement = "—"

    print(f"{prompt:<22} {ptype:<10} {r_gap:>12.4f} {b_gap:>13.4f} {improvement:>12}")

print("=" * 72)

valid = [(r, b) for (_, t, *_), r, b in zip(results, rewrite_gaps, baseline_gaps) if "Haiku" not in _]
valid_r = [r for r, b in valid]
valid_b = [b for r, b in valid]

avg_r = sum(valid_r) / len(valid_r)
avg_b = sum(valid_b) / len(valid_b)
overall = ((avg_r - avg_b) / avg_b) * 100

print(f"{'Avg':<22} {'':<10} {avg_r:>12.4f} {avg_b:>13.4f} {f'+{overall:.0f}%':>12}")
print(f"\n  Rewrite wins: {sum(r > b for r, b in zip(rewrite_gaps, baseline_gaps))}/6 cases")
print(f"  Average improvement: {overall:.1f}x larger gap")

Prompt                 Type        Rewrite Gap  Baseline Gap  Improvement
Gradient descent       In-dist          0.4085        0.2234         +83%
Precision vs Recall    In-dist          1.3984        0.1386        +909%
BERT vs GPT            In-dist          0.4907        0.1972        +149%
Haiku (ML)             OOD             -0.0601        0.0149       failed
Refrigerator           OOD              0.2290        0.0590        +288%
Australia capital      OOD              0.1472        0.0707        +108%
Avg                                     0.4356        0.1173        +271%

  Rewrite wins: 5/6 cases
  Average improvement: 271.4x larger gap


## 12. Export Models

In [20]:
import shutil
from google.colab import files

# Zip rewrite model
shutil.make_archive("reward_model_rewrite", "zip", "./reward_model/best_model")
files.download("reward_model_rewrite.zip")

# Zip baseline model
shutil.make_archive("reward_model_baseline", "zip", "./reward_model_baseline/best_model")
files.download("reward_model_baseline.zip")

# Zip dataset
shutil.make_archive("rlhf_dataset", "zip", "./rlhf_data")
files.download("rlhf_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>